# B2-023-generative-models-diffusion — Practice p18 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** denoising-diffusion-probabilistic-models

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

## Solution

**Plan.** Implement the p09 schedule and closed-form sampler, the p10 loss (which calls the module-level `q_sample` by name), and the pinned `Denoiser`.
Seed immediately before constructing the model, evaluate the held-out denoising loss once **before** training on its own fixed `t_eval`/`eps_eval`, train for exactly 1500 full-batch Adam updates with `t` then `eps` drawn from the dedicated generator inside `train_ddpm`, and evaluate the same held-out loss again.
For the duration of the training call only, the module-level `q_sample` is replaced by a recorder that keeps every `x0` it receives and then calls the original; it is restored before the held-out evaluation, which legitimately passes held-out rows through the same seam.

In [ ]:
import copy

DATASET = "mixture2d"
T = 50


def train_rows():
    return generative_data.train_tensor(DATASET)


def heldout_rows():
    return generative_data.heldout_tensor(DATASET)


def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def q_sample(x0, t, eps, alpha_bar):
    if x0.shape != eps.shape:
        raise ValueError("x0 and eps must have the same shape")
    if (not isinstance(t, torch.Tensor) or t.dtype != torch.int64 or t.dim() != 1
            or x0.dim() != 2 or t.shape[0] != x0.shape[0]):
        raise ValueError("t must be an int64 tensor of shape (B,)")
    T_max = alpha_bar.shape[0] - 1
    if bool((t < 1).any()) or bool((t > T_max).any()):
        raise ValueError("timesteps must lie in 1..T")
    ab = alpha_bar[t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


def ddpm_loss(model, x0, t, eps, alpha_bar):
    T_max = alpha_bar.shape[0] - 1
    x_t = q_sample(x0, t, eps, alpha_bar)            # module-level name, looked up at call time
    eps_hat = model(x_t, (t.to(x0.dtype) / T_max).unsqueeze(1))
    if eps_hat.shape != eps.shape:
        raise ValueError("model output must have the shape of eps")
    return ((eps_hat - eps) ** 2).mean()


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, x_t, t_scaled):
        return self.net(torch.cat([x_t, t_scaled], dim=1))


def train_ddpm(model, batch, optimizer):
    noise_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(1500):
        t = torch.randint(1, T + 1, (batch.shape[0],), generator=noise_generator)
        eps = torch.randn(batch.shape[0], 2, generator=noise_generator)
        optimizer.zero_grad(set_to_none=True)
        loss = ddpm_loss(model, batch, t, eps, SCHEDULE["alpha_bar"])
        loss.backward()
        optimizer.step()
        trace.append(loss.item())
    return trace


def split_hash_sets(name):
    hashes = generative_data.ROW_SHA256[name]
    return ({hashes[i] for i in generative_data.TRAIN_IDS[name]},
            {hashes[i] for i in generative_data.HELDOUT_IDS[name]})


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)
SCHEDULE = make_schedule(50, 1e-3, 0.2)

### Held-out loss before training, training with the seam recorded, held-out loss after

In [ ]:
torch.manual_seed(SEED)
model = Denoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
model_init = copy.deepcopy(model)

eval_generator = torch.Generator().manual_seed(SEED + 1)
t_eval = torch.randint(1, 51, (64,), generator=eval_generator)
eps_eval = torch.randn(64, 2, generator=eval_generator)
with torch.no_grad():
    heldout_before = ddpm_loss(model, heldout_rows(), t_eval, eps_eval, SCHEDULE["alpha_bar"]).item()

recorded = []
original_q_sample = q_sample


def recording_q_sample(x0, t, eps, alpha_bar):
    recorded.append(x0.detach().clone())
    return original_q_sample(x0, t, eps, alpha_bar)


q_sample = recording_q_sample
try:
    trace = train_ddpm(model, train_rows(), optimizer)
finally:
    q_sample = original_q_sample
seam_restored = q_sample is original_q_sample
recorded_calls = len(recorded)
recorded_rows = torch.unique(torch.cat(recorded), dim=0)
recorded_hashes = {generative_data.row_sha256(row) for row in recorded_rows}

with torch.no_grad():
    heldout_after = ddpm_loss(model, heldout_rows(), t_eval, eps_eval, SCHEDULE["alpha_bar"]).item()
baseline = eps_eval.pow(2).mean().item()

first50, last50 = sum(trace[:50]) / 50, sum(trace[-50:]) / 50
changed = any(not torch.equal(a, b) for a, b in zip(model.parameters(), model_init.parameters()))
print("loss window means every 250 steps:", [round(sum(trace[a:a + 50]) / 50, 4) for a in range(0, 1500, 250)])
print(f"first-50 mean {first50:.4f}  last-50 mean {last50:.4f}  ratio {last50 / first50:.3f}")
print(f"held-out before {heldout_before:.4f}  after {heldout_after:.4f}  zero-predictor baseline {baseline:.4f}")
print(f"alpha_bar[50] = {SCHEDULE['alpha_bar'][50].item():.6f};  q_sample calls {recorded_calls}, distinct x0 rows {recorded_rows.shape[0]}")

### Interpretation

The training loss levels off well above zero because at large $t$ the input $x_t$ is almost pure noise, so even the best possible network cannot recover the specific $\varepsilon$ from it: the minimum achievable mean squared error, averaged over uniformly drawn $t$, is strictly positive.
The held-out loss uses its own fixed `t_eval` and `eps_eval`, drawn once from a separate seed, so that the before and after values are computed on identical timesteps and noises (a paired comparison) and are independent of how many draws the training stream consumed.
(D) adds an absolute reference: the zero predictor's loss $\lVert\varepsilon\rVert^2$ is what a network that learned nothing about the data would get on exactly these draws, so beating half of it shows real denoising skill, whereas (C)'s comparison with the untrained network could be met merely by fixing a badly scaled initialization.

### Answer check

In [ ]:
assert train_rows().shape == (256, 2) and heldout_rows().shape == (64, 2)
# (A)
assert len(trace) == 1500 and all(isinstance(v, float) and math.isfinite(v) for v in trace)
# (B)
assert last50 <= 0.60 * first50
# (C)
assert heldout_after <= 0.40 and heldout_after <= 0.5 * heldout_before
# (D)
assert heldout_after <= 0.5 * baseline
# (E)
assert changed and SCHEDULE["alpha_bar"][50].item() < 0.005
# (F)
assert seam_restored and recorded_calls == 1500
assert not (recorded_hashes & HELDOUT_HASHES)
assert TRAIN_HASHES <= recorded_hashes